<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformando Datos: Feature Engineering - Creación de Custom Transformers

### Objetivo
Enseñar el diseño de clases personalizadas basadas en **FunctionTransformer** en Python para empaquetar lógica de negocio específica dentro de tuberías de Scikit-Learn compatibles con validación cruzada.

### Introducción: El porqué de los Transformadores Personalizados
¡Hola! Hoy vamos a dar un paso gigante en tu carrera como científico de datos. Vamos a aprender a capturar toda esa lógica de negocio que a veces queda suelta en tus notebooks y la vamos a transformar en piezas de ingeniería robustas y elegantes.

En proyectos reales, la limpieza de datos va más allá de borrar nulos. A menudo enfrentamos reglas complejas, como normalizar precios o calcular antigüedades. El gran problema es que, si aplicamos estas funciones manualmente antes del entrenamiento, rompemos el flujo de Scikit-Learn. Esto genera **Data Leakage** (fuga de datos), pues la lógica de preprocesamiento podría contaminar los pliegues de validación.

Al usar transformadores personalizados, garantizamos que cada transformación se aplique de forma aislada y reproducible, tratando a nuestra lógica como un **ciudadano de primera clase** dentro de una Pipeline automatizada.

### 1. Preparación del Escenario y Datos de Prueba
Imagina el caso de Mateo, un analista de seguros. Él recibe montos de siniestros como cadenas de texto con símbolos de moneda que Python no entiende. Para automatizar su flujo, primero necesitamos crear estos datos de prueba y definir la función de limpieza.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV

# Simulación de datos de Mateo (empresa de seguros)
data = {
    'monto': ['$1.250,50', '$3.400,00', '$850,25', '$5.000,00', '$2.100,75'],
    'antiguedad_cliente': [5, 2, 1, 10, 4],
    'objetivo': [100, 250, 60, 400, 180]
}

df = pd.DataFrame(data)
print('Datos Crudos recibidos por Mateo:')
print(df)
print()

### 2. El puente: FunctionTransformer
La clase **FunctionTransformer** actúa como un puente. Toma cualquier función normal de Python y le otorga los métodos **fit** y **transform**. Esto es vital porque las Pipelines esperan que todos sus componentes hablen el mismo idioma, incluso si la transformación no necesita aprender nada de los datos (como un escalado).

In [ ]:
def limpiar_moneda(X):
    # Creamos una copia para no afectar el dataframe original
    X_copy = X.copy()
    # Si es un DataFrame, trabajamos sobre la columna específica
    if isinstance(X_copy, pd.DataFrame):
        col = X_copy['monto'].astype(str)
        col = col.str.replace('$', '', regex=False)
        col = col.str.replace('.', '', regex=False)
        col = col.str.replace(',', '.', regex=False)
        X_copy['monto'] = col.astype(float)
    return X_copy

# Convertimos nuestra función en un transformador de Scikit-Learn
transformer_moneda = FunctionTransformer(limpiar_moneda)

# Aplicamos la transformación
df_limpio = transformer_moneda.transform(df)

print('Datos después de pasar por FunctionTransformer:')
print(df_limpio)
print()
print('Tipo de dato de la columna monto:', df_limpio['monto'].dtype)

### 3. Integración en una Pipeline y Validación Cruzada
La verdadera magia ocurre cuando integramos este transformador en un flujo de trabajo profesional. Esto permite que el sistema evalúe el rendimiento del modelo asegurando que la limpieza ocurra de forma independiente en cada pliegue de la validación cruzada, evitando el sesgo del analista.

In [ ]:
# Definimos las variables de entrada y salida
X = df[['monto', 'antiguedad_cliente']]
y = df['objetivo']

# Creamos la Pipeline: Limpieza -> Modelo
pipeline_seguros = Pipeline([
    ('limpieza_negocio', FunctionTransformer(limpiar_moneda)),
    ('regresor', LinearRegression())
])

# Configuramos un GridSearchCV sencillo para demostrar compatibilidad
# Aunque el regresor es simple, esto permite optimizar cualquier parámetro
param_grid = {
    'regresor__fit_intercept': [True, False]
}

grid_search = GridSearchCV(pipeline_seguros, param_grid, cv=2)
grid_search.fit(X, y)

print('Resultados de la Validación Cruzada:')
resultados = pd.DataFrame(grid_search.cv_results_)
print(resultados[['param_regresor__fit_intercept', 'mean_test_score']])
print()
print('¡Pipeline ejecutada con éxito!')

### Conclusión
¿Te das cuenta de la elegancia de este enfoque? Al final del día, estamos construyendo software, no solo haciendo gráficas. Al empaquetar la lógica de negocio en transformadores, tus modelos son menos propensos a errores humanos y tus compañeros pueden reutilizar tu código con una sola línea.

Dominar los transformadores personalizados es lo que separa a un principiante de un verdadero ingeniero de Machine Learning. **¡A seguir practicando!**